**No meaningful metadata shortcut: DICOM headers reach only 0.598 macro AUC across unseen scanners**

Public LB scores passed 0.8 and 0.9 within about a day of launch. That seemed strangely high for this label set in such short period, so I tested whether DICOM metadata alone could account for this.

It can't. Full header metadata, no pixels, reaches 0.6515 macro AUC under random folds and 0.5981 under scanner-grouped folds. The 0.053 increment is site memorization and it does not transfer to unseen scanners.

**Method**

Two probes, with the decision threshold written down before running anything:

Probe A: site identifiability. Cluster studies on Manufacturer + ManufacturerModelName + SoftwareVersions + ImagingFrequency + ReceiveCoilName. No labels needed. Result: 265 distinct fingerprints, top 20 covering 45.5% of studies.

Probe B: metadata -> targets. HistGradientBoosting on study-level metadata features, targets = report-derived labels, scored under random 5-fold and under GroupKFold on the fingerprint above. The gap is the number of interest.

Features: series composition (plane × Fluid_Sensitive × Fat_Suppression counts, protocol signature string), plus per-study median/min/max of TR, TE, TI, flip angle, echo train length, pixel bandwidth, averages, phase encoding steps, slice thickness, spacing, pixel spacing, matrix size, field strength, ImagingFrequency, and categorical scanner/coil/software identity.

**Results:**

| Label            | Random | Site-grouped |  Drop |
| ---------------- | --------: | -----------: | ----: |
| ACL              |  0.705    |        0.670 | 0.035 |
| MCL              |  0.683    |        0.648 | 0.035 |
| Medial Meniscus  |  0.590    |        0.548 | 0.042 |
| Lateral Meniscus |  0.595    |        0.565 | 0.030 |
| Medial OA        |  0.652    |        0.578 | 0.074 |
| Lateral OA       |  0.637    |        0.563 | 0.074 |
| PF OA            |  0.680    |        0.599 | 0.081 |
| Effusion         |  0.628    |        0.582 | 0.046 |
| Synovitis        |  0.644    |        0.602 | 0.042 |
| Baker's          |  0.765    |        0.717 | 0.048 |
| Contusion        |  0.633    |        0.587 | 0.046 |
| Fracture         |  0.605    |        0.519 | 0.086 |
| **Macro**        |**0.6516** |   **0.5981** | **0.0534** |


Series composition alone (no DICOM reads at all) gives 0.5954. So the entire DICOM header pass adds 0.056 over the four columns already in `train_series.csv`

**Note:**
- Targets are report-derived, not expert annotations. Only 58 studies carry per-condition labels, which is too few to fit or validate against. So part of the 0.053 may be metadata predicting reporting style rather than disease.
- 265 fingerprints is finer-grained than institution. I'm separating individual scanners and software revisions within sites, so the grouped folds are stricter than true site holdout. That makes 0.053 an upper bound for a second, independent reason.


The report extractor used to generate targets is from: https://www.kaggle.com/code/romanrozen/rsna-knee-data-structure-eda-baseline

In [1]:
import numpy as np
import pandas as pd
import pydicom
import os, re, time, warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from __future__ import annotations
import re
import unicodedata
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.metrics import roc_auc_score


warnings.filterwarnings("ignore")

In [2]:
T0 = time.time()
def log(m): print(f"[{time.time()-T0:6.1f}s] {m}", flush=True)

def find_root():
    for c in [Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
              Path("/kaggle/input/rsna-knee-abnormality-detection")]:
        if (c / "train.csv").is_file():
            return c
    for p in Path("/kaggle/input").rglob("train.csv"):
        if (p.parent / "train_series.csv").is_file():
            return p.parent
    raise FileNotFoundError("competition mount not found")

ROOT = find_root()
train        = pd.read_csv(ROOT / "train.csv")
train_series = pd.read_csv(ROOT / "train_series.csv")
test         = pd.read_csv(ROOT / "test.csv")
test_series  = pd.read_csv(ROOT / "test_series.csv")

TARGETS = ["ACL","MCL","Medial Meniscus","Lateral Meniscus","Medial OA",
           "Lateral OA","PF OA","Effusion","Synovitis","Baker's","Contusion","Fracture"]

gold = train[train[TARGETS].notna().all(axis=1)]
log(f"root {ROOT}")
log(f"train {train.shape}  series {train_series.shape}  gold {len(gold)}")

[   0.3s] root /kaggle/input/competitions/rsna-knee-abnormality-detection
[   0.3s] train (4407, 14)  series (24371, 5)  gold 58


In [3]:
def sample_dicoms(split, n=12):
    base, out = ROOT / split, []
    if not base.is_dir(): return out
    for study in sorted(os.scandir(base), key=lambda e: e.name):
        if not study.is_dir(): continue
        for series in sorted(os.scandir(study.path), key=lambda e: e.name):
            if not series.is_dir(): continue
            f = [e.name for e in os.scandir(series.path) if e.name.endswith(".dcm")]
            if f: out.append(os.path.join(series.path, sorted(f)[len(f)//2]))
            if len(out) >= n: return out
    return out

def tags_of(path):
    ds = pydicom.dcmread(path, stop_before_pixels=True, force=True)
    return {(e.keyword or str(e.tag)) for e in ds if e.keyword != "PixelData"}

tr_tags = set().union(*[tags_of(p) for p in sample_dicoms("train_series")])
te_tags = set().union(*[tags_of(p) for p in sample_dicoms("test_series")])

print(f"train tags {len(tr_tags)} | test tags {len(te_tags)}\n")
print("IN TEST (usable at inference):")
print(sorted(te_tags))
print("\nTRAIN ONLY (signal you can't deploy, but still shapes training):")
print(sorted(tr_tags - te_tags))

train tags 67 | test tags 64

IN TEST (usable at inference):
['AcquisitionMatrix', 'AcquisitionNumber', 'BitsAllocated', 'BitsStored', 'BodyPartExamined', 'Columns', 'ContrastBolusAgent', 'EchoNumbers', 'EchoTime', 'EchoTrainLength', 'FlipAngle', 'FrameOfReferenceUID', 'HighBit', 'ImageOrientationPatient', 'ImagePositionPatient', 'ImageType', 'ImagingFrequency', 'InPlanePhaseEncodingDirection', 'InStackPositionNumber', 'InstanceNumber', 'InversionTime', 'Laterality', 'MRAcquisitionType', 'MagneticFieldStrength', 'Manufacturer', 'ManufacturerModelName', 'Modality', 'NumberOfAverages', 'NumberOfPhaseEncodingSteps', 'NumberOfTemporalPositions', 'PatientID', 'PatientPosition', 'PatientSex', 'PercentPhaseFieldOfView', 'PercentSampling', 'PhotometricInterpretation', 'PixelBandwidth', 'PixelRepresentation', 'PixelSpacing', 'ReceiveCoilName', 'RepetitionTime', 'RescaleIntercept', 'RescaleSlope', 'Rows', 'SOPClassUID', 'SOPInstanceUID', 'SamplesPerPixel', 'ScanOptions', 'ScanningSequence', 'Seq

In [4]:
# https://www.kaggle.com/code/romanrozen/rsna-knee-data-structure-eda-baseline
TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture",
]


# Turkish dotted/dotless i must be folded before casefolding, otherwise "İZLENMEZ"
# and "izlenmez" diverge. ß and the Croatian/Serbian d-with-stroke likewise.
_PRE = str.maketrans({
    "ı": "i", "İ": "i", "I": "i", "ß": "ss", "đ": "d", "Đ": "d",
    "ø": "o", "Ø": "o", "æ": "ae", "Æ": "ae",
})


def normalize(text: str) -> str:
    """Fold case, diacritics and separators; keep Greek and Cyrillic letters.

    NFKD decomposition strips Latin accents and Greek tonos alike (ά -> α), which is what
    we want: reports are inconsistent about accents. It also maps the MICRO SIGN U+00B5
    to a real mu, which matters because most Greek reports here use the wrong codepoint.
    """
    if not isinstance(text, str):
        return ""
    text = text.translate(_PRE).lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = text.replace("­", "")                    # soft hyphen
    text = re.sub(r"[_\-/\\]+", " ", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text


_SENT_SPLIT = re.compile(r"(?<=[.;!?])\s+|\n+")


def clauses(text: str):
    """Split into clauses, then attach `header:` lines to the value that follows.

    A report line reading `Fractures :` followed by `Aucune.` is one statement. Splitting
    on punctuation alone separates the anatomy from its negation and flips the label.
    """
    norm = normalize(text)
    raw = [c.strip() for c in _SENT_SPLIT.split(norm) if c and c.strip()]

    merged = []
    for i, c in enumerate(raw):
        # A fragment ending in a colon is a heading for the next fragment. Structured
        # English reports write long ones - "lateral compartment (meniscus, collateral
        # ligament complex, cartilage):" is eight words - so the cap is generous, and
        # the heading is kept as its own clause too in case it carries the finding.
        if c.endswith(":") and len(c.split()) <= 14 and i + 1 < len(raw):
            merged.append(c + " " + raw[i + 1])
        merged.append(c)
    # Comma-separated enumerations inside a long clause hide separate assertions.
    out = []
    for c in merged:
        out.append(c)
        if len(c.split()) > 25:
            out.extend(p.strip() for p in c.split(",") if len(p.split()) > 2)
    return out


def _rx(*alts: str) -> re.Pattern:
    return re.compile("|".join(alts))


In [5]:
# https://www.kaggle.com/code/romanrozen/rsna-knee-data-structure-eda-baseline
NEGATION = _rx(
    # en
    r"\bno\b", r"\bnot\b", r"\bwithout\b", r"\bnegative for\b", r"\babsence\b",
    r"\bno evidence\b", r"\bunremarkable\b", r"\bfree of\b",
    # es
    r"\bsin\b", r"\bno hay\b", r"\bausencia\b", r"\bausentes?\b",
    # fr
    r"\bpas de\b", r"\bsans\b", r"\baucune?\b", r"\babsence\b",
    # nl
    r"\bgeen\b", r"\bzonder\b", r"\bniet\b",
    # de
    r"\bkeine?\b", r"\bohne\b", r"\bnicht\b",
    # tr
    r"\byok\b", r"\byoktur\b", r"izlenmemekte", r"saptanmadi", r"\bdegil\b",
    r"gozlenmemekte", r"mevcut degil", r"eslik etmiyor", r"\bizlenmedi\b",
    # hr / sr / bs
    r"\bnema\b", r"\bbez\b", r"\bnisu\b", r"\bnije\b",
    # el (accents already stripped)
    r"\bδεν\b", r"\bχωρις\b", r"ουδεν",
    # bg / ru
    r"\bбез\b", r"\bне\b", r"липсва", r"\bняма\b",
)

NORMALITY = _rx(
    r"\bnormal", r"\bintact\b", r"\bpreserved\b", r"\bwithin normal limits\b",
    r"limites normales", r"\bconservad", r"\bintegr", r"\bnormales\b",
    r"\bdoga(l|ll)\b", r"korunmus", r"\bnormaldir\b", r"olagan",
    r"\buredn", r"\bocuvan", r"\bodrzan", r"\bintakt",
    r"φυσιολογικ", r"ακεραι",
    r"unauffallig", r"regelrecht", r"\bintakt\b",
    r"нормал", r"запазен", r"съхранен", r"\bбез особености\b",
    r"\bgaaf\b", r"\bnormaal\b",
)

UNCERTAIN = _rx(
    r"\bpossible\b", r"\bprobable\b", r"\bsuspicious\b", r"\bsuspected\b",
    r"cannot (be )?exclude", r"\bmay\b", r"\bquestionable\b", r"\bequivocal\b",
    r"\bposible\b", r"sin criterios categoricos", r"\bdudos",
    r"\bmuhtemel\b", r"\bolasi\b", r"\bsupheli\b", r"\bizlenim",
    r"\bmoguce\b", r"\bvjerojatno\b", r"\bsumnja\b",
    r"πιθαν", r"υποπτ",
    r"\bmoglich", r"\bverdachtig", r"\bfraglich", r"\bV\.a\.\b",
    r"\bвъзможно\b", r"\bвероятно\b", r"суспект",
    r"\bmogelijk\b", r"\bverdacht\b",
)

# Pathology vocabulary shared by the paired rules.
TEAR = _rx(
    r"\btear", r"\btorn\b", r"\brupture", r"\bdisruption\b", r"discontinuit",
    r"\bavuls",
    r"\brotura\b", r"\broturas\b", r"\bruptura", r"\bdesgarro", r"\broto\b",
    r"\bdechirure", r"\bdechire",
    r"\bscheur", r"\bruptuur", r"gescheurd",
    r"\briss\b", r"einriss", r"\bruptur", r"zerreiss", r"\blasion",
    r"\byirtik", r"\byirtig", r"\bkopma\b", r"butunluk kaybi", r"\brupturu\b",
    r"\bpuknuce", r"\bruptur", r"\bprekid\b", r"\bpukotin",
    r"ρηξη", r"ρηξις", r"ρηγμα",
    r"руптура", r"разкъсв", r"разрив", r"скъсв",
)

DEGEN = _rx(
    r"degenerat", r"\bmucoid\b", r"\bmyxoid\b", r"\bfray", r"\bfissur",
    r"dejeneratif", r"\bmukoid\b", r"degenerativn", r"εκφυλιστ", r"дегенерат",
    r"\bmuco ?ide\b", r"aufgefasert",
)

INJURY = _rx(
    r"\binjur", r"\bsprain", r"\blesion", r"\blasion", r"\bedema\b", r"\boedema\b",
    r"\bodem\b", r"\bedem\b", r"\bοιδημα", r"\bодем", r"\bедем", r"\bstrain\b",
    r"\bhigh signal\b", r"\bsignal alteration\b", r"\bhiperintens", r"\bhyperintens",
    r"\bthicken", r"\bzadebljanje\b", r"\bverdikking\b", r"\bdistenzij",
    r"\blaksite\b", r"\blaxity\b", r"\bpartial\b", r"\bparcijaln", r"\bparcial",
    r"\bpartiel", r"\bpartiell",
)
ANAT = {
    "ACL": _rx(
        r"anterior cruciate", r"\bacl\b",
        r"cruzado anterior", r"\blca\b",
        r"croise anterieur",
        r"voorste kruisband", r"\bvkb\b",
        r"vorderes kreuzband", r"vorderen kreuzband", r"vordere kreuzband",
        r"on capraz", r"\bocb\b",
        r"prednji krizni", r"prednjeg krizn",
        r"προσθι[οα][^ ]* χιαστ", r"προσθιου χιαστου", r"χιαστο[^ ]* συνδεσμ",
        r"предна кръстна", r"предната кръстна",
        # Plural, unqualified: reports routinely clear both cruciates in one clause
        # ("Ligamentos cruzados y colaterales dentro de limites normales"). Without
        # this, Spanish ACL was silent on 88% of its reports and Dutch on 70%.
        r"cruciate ligaments", r"ligamentos cruzados", r"ligaments croises",
        r"kruisbanden", r"kreuzbander", r"capraz baglar", r"krizn[a-z]* ligament[a-z]*",
        r"χιαστοι συνδεσμ", r"χιαστων συνδεσμ", r"кръстните връзки", r"кръстни връзки",
    ),
    "MCL": _rx(
        r"medial collateral", r"\bmcl\b", r"tibial collateral",
        r"colateral medial", r"colateral interno", r"\blcm\b",
        r"collateral medial", r"collateral interne",
        r"mediale collaterale", r"binnenband",
        r"innenband", r"mediales? kollateral",
        r"\bic yan bag", r"medial kollateral", r"\biyb\b",
        r"medijalni kolateraln", r"medijalnog kolateraln",
        r"εσω πλαγι", r"εσωτερικο πλαγι",
        r"медиален колатерал", r"вътрешна странична",
        # Same plural pattern as the cruciates.
        # "Ligamentos cruzados y colaterales" separates the noun from its adjective, so
        # the adjective has to stand alone as a cue.
        r"\bcolaterales\b", r"\bcollateraux\b", r"\bcollateralen\b", r"\bkolateralni\b",
        r"collateral ligaments", r"ligamentos colaterales", r"ligaments collateraux",
        r"collaterale banden", r"kollateralbander", r"seitenbander", r"yan baglar",
        r"kolateraln[a-z]* ligament[a-z]*", r"πλαγιοι συνδεσμ", r"πλαγιων συνδεσμ",
        r"колатерални връзки", r"страничните връзки",
    ),
    "Medial Meniscus": _rx(
        r"medial meniscus", r"\bmm\b(?= tear)", r"medial menisc",
        r"menisco medial", r"menisco interno",
        r"menisque medial", r"menisque interne",
        r"mediale meniscus", r"binnenmeniscus",
        r"innenmeniskus", r"medialen? meniskus", r"innenmeniskushinterhorn",
        r"medyal menisk", r"\bic menisk",
        r"medijalni meniskus", r"medijalnog meniskusa", r"medijalnom meniskusu",
        r"εσω μηνισκ", r"μηνισκ[^ ]* του εσω", r"εσω διαμερισμα[^.]{0,40}μηνισκ",
        r"медиалния менискус", r"медиален менискус", r"вътрешния менискус",
    ),
    "Lateral Meniscus": _rx(
        r"lateral meniscus", r"lateral menisc",
        r"menisco lateral", r"menisco externo",
        r"menisque lateral", r"menisque externe",
        r"laterale meniscus", r"buitenmeniscus",
        r"aussenmeniskus", r"lateralen? meniskus",
        r"lateral menisk", r"\bdis menisk",
        r"lateralni meniskus", r"lateralnog meniskusa", r"lateralnom meniskusu",
        r"εξω μηνισκ", r"μηνισκ[^ ]* του εξω", r"εξω διαμερισμα[^.]{0,40}μηνισκ",
        r"латералния менискус", r"латерален менискус", r"външния менискус",
    ),
}

# Osteoarthritis is rarely written as "osteoarthritis". It is written as cartilage loss,
# chondropathy grade, joint space narrowing, or osteophytes - scoped to a compartment.
OA_EVIDENCE = _rx(
    r"osteoarthrit", r"\barthros", r"\bgonarthros", r"\bosteoarthros",
    r"chondropath", r"chondromalac", r"condropat", r"condromalac",
    r"cartilage loss", r"cartilage thinning", r"chondral (loss|defect|ulcer|thinning)",
    r"osteophyt", r"osteofit", r"osteofyt", r"osteofito", r"osteophyten",
    r"joint space narrowing", r"pinzamiento articular",
    r"kikirdak kayb", r"kikirdak incelme", r"kondropati", r"kondral",
    r"kraakbeen(lijden|verlies)", r"gonartrose", r"artrose",
    r"knorpel(verlust|schaden|defekt)", r"arthrose", r"gonarthrose",
    r"hrskavic", r"hondromalac", r"artroz", r"osteoartrit",
    r"χονδρ[^ ]*παθ", r"αρθριτ", r"αρθρωσ", r"οστεοφυτ",
    r"αρθρικου χονδρου", r"εξαλειψη του αρθρικου χονδρου",
    r"артроз", r"хондропат", r"остеофит", r"хрущял[^.]{0,30}(изтън|увред|дефект)",
    r"ulcera[s]? condral", r"cartilago[^.]{0,25}(perdida|adelgaz)",
    r"icrs grade", r"outerbridge",
)

COMPARTMENT = {
    "Medial OA": _rx(
        r"medial (femorotibial|tibiofemoral|compartment)",
        r"compartimento femorotibial medial", r"femorotibial interno",
        r"mediaal femorotibiaal", r"mediale femorotibial",
        r"medial femorotibial", r"medialen kompartiment", r"innere[sn]? kompartiment",
        r"medyal femorotibial", r"ic kompartman", r"medyal kompartman",
        r"medijaln[^ ]* (femorotibi|odjelj|kompartm)",
        r"εσω διαμερισμα", r"εσω κνημιαι", r"εσω μηριαι",
        r"медиалн[^ ]* (компартм|отдел|тибиал|феморотиб)",
        r"medial (femoral|tibial) (condyle|plateau)", r"condilo femoral medial",
        r"medialen? (femurkondyl|tibiaplateau)", r"mediale femorale condyl",
    ),
    "Lateral OA": _rx(
        r"lateral (femorotibial|tibiofemoral|compartment)",
        r"compartimento femorotibial lateral", r"femorotibial externo",
        r"lateraal femorotibiaal", r"laterale femorotibial",
        r"lateral femorotibial", r"lateralen kompartiment", r"aussere[sn]? kompartiment",
        r"lateral femorotibial", r"dis kompartman", r"lateral kompartman",
        r"lateraln[^ ]* (femorotibi|odjelj|kompartm)",
        r"εξω διαμερισμα", r"εξω κνημιαι", r"εξω μηριαι",
        r"латералн[^ ]* (компартм|отдел|тибиал|феморотиб)",
        r"lateral (femoral|tibial) (condyle|plateau)", r"condilo femoral lateral",
        r"lateralen? (femurkondyl|tibiaplateau)", r"laterale femorale condyl",
    ),
    "PF OA": _rx(
        r"patellofemoral", r"femoropatellar", r"femoropatelar", r"patelofemoral",
        r"retropatellar", r"retrorotulian", r"\btrochlea", r"\btroclea", r"\btroklea",
        r"\bpatella\b", r"\bpatellar\b", r"\brotulian", r"\brotula\b", r"\bpatele\b",
        r"\bpatellae?\b", r"patellofemoraal", r"femoropatellair",
        r"επιγονατιδ", r"μηροεπιγονατιδ", r"τροχιλ",
        r"пател", r"феморопател", r"тролх",
        r"anterior compartment", r"compartimento anterior", r"prednj[^ ]* odjeljk",
    ),
}

# Self-declaring findings: the term itself is the finding.
DIRECT = {
    "Effusion": _rx(
        r"\beffusion", r"joint fluid", r"intra ?articular fluid", r"\bhydrops\b",
        r"derrame articular", r"\bderrame\b", r"liquido articular",
        r"epanchement",
        r"gewrichtsvocht", r"\bvocht\b", r"\bhydrops\b", r"gewrichtseffusie",
        r"gelenkerguss", r"\berguss\b", r"gelenksergu",
        # "diz eklemi ici sivi miktari ... artmis" and "eklem icerisinde yaygin sivi
        # artisi" both occur; the noun takes a possessive suffix, so `eklem ` alone
        # misses. Match the stem plus any suffix.
        r"eklem\w* ic\w* sivi", r"efuzyon", r"eklem sivisi",
        r"sivi (miktari|artisi|birikimi)", r"sivi artis", r"\bsivi\b[^.]{0,25}artmis",
        r"\bizljev", r"\bizliv", r"zglobn[^ ]* tekucin", r"\bhidrops\b",
        r"αρθρικ[^ ]* υγρ", r"υγρου ενδαρθρικα", r"ενδαρθρικ[^ ]* υγρ", r"ποσοτητα υγρου",
        r"ενδαρθρικ", r"αρθρικη συλλογη", r"υγρο στην αρθρωση", r"υγρου στην αρθρωση",
        r"ставен излив", r"излив", r"ставна течност", r"синовиална течност",
    ),
    "Synovitis": _rx(
        r"synovit", r"sinovit", r"synovial (thickening|proliferation|hypertroph)",
        r"synovitis", r"synoviale? (verdikking|proliferatie)",
        r"synovialitis", r"synovialis(verdickung|proliferation)",
        r"sinovijalitis", r"sinovitis", r"zadebljanje sinovij",
        r"υμενιτιδα", r"συνοβιτιδα", r"υμενικ[^ ]* υπερτροφ", r"αρθρικου υμεν",
        r"синовит", r"синовиал[^ ]* (задебел|пролифер)",
        r"verdikkingen van (het )?synovium", r"pannus",
    ),
    "Baker's": _rx(
        r"baker", r"popliteal cyst", r"quiste popliteo", r"quistes popliteos",
        r"kyste poplite", r"popliteale? cyst", r"poplitealzyste", r"bakerzyste",
        r"popliteal kist", r"\bbakerova\b", r"poplitealn[^ ]* cist",
        r"κυστη baker", r"πολυχωρη συνοβιακη κυστη", r"κυστη του baker",
        r"киста на бейкър", r"бейкърова киста", r"поплитеална киста",
        r"gastrocnemio ?semimembranos", r"gastrocnemius semimembranosus burs",
    ),
    "Contusion": _rx(
        r"\bcontusion", r"bone bruise", r"bone marrow (o?edema|contusion)",
        r"\bkontuz", r"medular bone o?edema", r"marrow o?edema",
        r"contusion osea", r"edema oseo", r"edema de medula osea",
        r"oedeme osseux", r"contusion osseuse",
        r"botcontusie", r"botoedeem", r"beenmergoedeem", r"botmergoedeem",
        r"knochenmarkodem", r"knochenodem", r"kontusion", r"bone bruise",
        r"kemik kontuzyonu", r"kemik iligi odemi", r"kemik odemi",
        r"kostani edem", r"edem kosti", r"kontuzij",
        r"οστεομυελικ[^ ]* οιδημα", r"οστικο οιδημα", r"μυελικο οιδημα",
        r"костномозъчен едем", r"костен едем", r"контузионен",
    ),
    "Fracture": _rx(
        r"\bfractur", r"\bfract\b",
        r"\bfractura", r"\bfracturas\b",
        r"\bfractuur", r"\bbreuk\b",
        r"\bfraktur", r"\bbruch\b",
        r"\bkirik\b", r"\bkirigi\b", r"\bkirik\b",
        r"\bfraktur", r"\bprijelom", r"impresijsk[^ ]* fraktur",
        r"καταγμα", r"καταγματ",
        r"фрактур", r"счупван", r"фисур",
        r"insufficiency fracture", r"stress fracture", r"avulsion fracture",
        r"subchondral fracture", r"subkondral kiri",
    ),
}

# Terms that look like a finding but are not the finding being scored.
DECOY = {
    "Fracture": _rx(r"no fracture", r"microfractur", r"\bfracture (risk|prophyla)"),
    "Baker's": _rx(r"meniscal cyst", r"quiste meniscal", r"ganglion"),
}

PAIRED = {"ACL", "MCL", "Medial Meniscus", "Lateral Meniscus"}
OA_TARGETS = {"Medial OA", "Lateral OA", "PF OA"}

STEM_MENISCUS = _rx(r"menisc\w*", r"menisk\w*", r"μηνισκ\w*", r"мениск\w*")
STEM_CRUCIATE = _rx(r"cruciate", r"cruzado", r"croise", r"kruisband", r"kreuzband",
                    r"capraz bag\w*", r"krizn\w*", r"χιαστ\w*", r"кръстн\w*",
                    r"\bacl\b", r"\bpcl\b", r"\blca\b", r"\blcp\b", r"\bvkb\b",
                    r"\bhkb\b", r"\bocb\b", r"\bacb\b")
STEM_COLLATERAL = _rx(r"collateral\w*", r"colateral\w*", r"kollateral\w*",
                      r"collaterale\w*", r"kolateraln\w*", r"yan bag\w*",
                      r"πλαγι\w*", r"колатерал\w*", r"странич\w*",
                      r"innenband\w*", r"aussenband\w*", r"binnenband\w*",
                      r"\bmcl\b", r"\blcl\b", r"\blcm\b", r"\biyb\b")

SIDE_MEDIAL = _rx(r"\bmedial\w*", r"\bmedyal\w*", r"\bmedijaln\w*", r"\bmediaal\w*",
                  r"\bmediale\w*", r"\binterno\w*", r"\binterne\w*", r"\binnen\w*",
                  r"\bic\b", r"\bunutarnj\w*", r"\bεσω\w*", r"\bεσωτερικ\w*",
                  r"\bмедиал\w*", r"\bвътреш\w*", r"\btibial collateral\b",
                  r"\bbinnen\w*", r"\bmediaal\b")
SIDE_LATERAL = _rx(r"\blateral\w*", r"\bexterno\w*", r"\bexterne\w*", r"\bdis\b",
                   r"\blateraln\w*", r"\baussen\w*", r"\bbuiten\w*", r"\bεξω\w*",
                   r"\bεξωτερικ\w*", r"\bлатерал\w*", r"\bвъншн\w*",
                   r"\bfibular collateral\b", r"\bvanjsk\w*")
SIDE_ANTERIOR = _rx(r"\banterior\w*", r"\bant\b", r"\bon\b", r"\bprednj\w*",
                    r"\bvorder\w*", r"\bvoorste\b", r"\bπροσθι\w*", r"\bпредн\w*",
                    r"\banteriyor\w*", r"\bavant\b", r"\bant[eé]rieur\w*")

# Fracture is the target whose stem varies most across the corpus.
STEM_FRACTURE = _rx(r"fractur\w*", r"fraktur\w*", r"fractuur\w*", r"\bfract\b",
                    r"kiri[kgğ]\w*", r"prijelom\w*", r"lom kosti", r"\bbreuk\w*",
                    r"\bbruch\w*", r"καταγμα\w*", r"καταγματ\w*", r"фрактур\w*",
                    # NOT a bare `fissur\w*`: "fisuras condrales" and "full thickness
                    # fissures in the articular cartilage" describe cartilage, not bone.
                    # The stem has to be anchored to a bone word to mean a fracture.
                    r"счупван\w*", r"fisur\w* (osea|oseas|kost)", r"fissur\w* kost")

STEM_OA_COMPARTMENT = _rx(r"compartment\w*", r"compartimento\w*", r"compartiment\w*",
                          r"kompartman\w*", r"kompartiment\w*", r"odjelj\w*",
                          r"διαμερισμα\w*", r"компартм\w*", r"\bотдел\w*",
                          r"femorotibial\w*", r"femorotibiaal\w*", r"tibiofemoral\w*",
                          r"femoro tibial\w*", r"κνημιαι\w*", r"μηριαι\w*",
                          r"femoral condyl\w*", r"tibial plateau\w*",
                          r"condilo femoral", r"platillo tibial", r"tibiaplateau\w*",
                          r"femurkondyl\w*", r"femoralne? kondil\w*",
                          r"tibijaln\w* plato", r"femoral kondil\w*",
                          r"tibia plato", r"tibyal plato")


def _near(clause: str, stem_rx: re.Pattern, qual_rx: re.Pattern, window: int = 55):
    """True if a stem match has a qualifier within `window` characters either side.

    Character windows rather than token windows, because word order differs: English
    puts the side before the noun, Greek and Bulgarian often after, and Turkish
    attaches it as a separate preceding adjective.
    """
    for m in stem_rx.finditer(clause):
        lo = max(0, m.start() - window)
        hi = min(len(clause), m.end() + window)
        if qual_rx.search(clause[lo:hi]):
            return True
    return False


# concept -> (stem, side) pairs used in addition to the phrase lexicons above
STEM_RULES = {
    "ACL": (STEM_CRUCIATE, SIDE_ANTERIOR),
    "MCL": (STEM_COLLATERAL, SIDE_MEDIAL),
    "Medial Meniscus": (STEM_MENISCUS, SIDE_MEDIAL),
    "Lateral Meniscus": (STEM_MENISCUS, SIDE_LATERAL),
    "Medial OA": (STEM_OA_COMPARTMENT, SIDE_MEDIAL),
    "Lateral OA": (STEM_OA_COMPARTMENT, SIDE_LATERAL),
}

SEV_LOW = _rx(
    r"\bsmall\b", r"\bminimal\b", r"\btrace\b", r"\bmild\b", r"\bslight\b",
    r"\btiny\b", r"\bscant\b", r"\bmimimal\b", r"\bdiscrete\b", r"\bfocal\b",
    r"\bleve\b", r"\bminim", r"\bpeque", r"\bligero\b", r"\bescaso\b", r"\bdiscreto\b",
    r"\bhafif\b", r"\bminimal\b", r"\baz miktarda\b", r"\bsilik\b",
    r"\bmanja\b", r"\bmanji\b", r"\bblago\b", r"\bdiskretn", r"\bmalo\b",
    r"\bgering", r"\bdiskret", r"\bkleine?r?\b", r"\bwenig\b", r"\bzarte?\b",
    r"\bbeperkte?\b", r"\bgeringe\b", r"\bweinig\b", r"\blichte?\b",
    r"\bηπι", r"\bμικρ", r"\bελαχιστ",
    r"\bминимал", r"\bлек", r"\bмалк", r"\bнеголям",
)

SEV_HIGH = _rx(
    r"\blarge\b", r"\bmarked\b", r"\bmassive\b", r"\bsevere\b", r"\bextensive\b",
    r"\bmoderate\b", r"\bgross\b", r"\bsignificant\b", r"\babundant\b", r"\btense\b",
    r"\bmoderad", r"\bimportante\b", r"\bsevera?\b", r"\bmarcad", r"\bcuantios",
    r"\bbelirgin\b", r"\byaygin\b", r"\bileri\b", r"\bciddi\b", r"\bbol\b",
    r"\bopsezan\b", r"\bveliki\b", r"\bizrazit", r"\bznacajn", r"\bumjeren",
    r"\bausgepragt", r"\bdeutlich", r"\bmassiv", r"\bmassig", r"\bgross",
    r"\buitgebreid", r"\bgevorderd", r"\bveel\b", r"\bmatige?\b",
    r"\bμετρι", r"\bμεγαλ", r"\bεκτεταμεν", r"\bευμεγεθ", r"\bσοβαρ",
    r"\bголям", r"\bизразен", r"\bзначим", r"\bумерен", r"\bобилен",
)

# OA is often asserted for the whole joint rather than per compartment
# ("tricompartmental osteoarthritis", "gonarthrose", "incipient OA of all three
# compartments"). Those statements are evidence for all three OA targets.
GLOBAL_OA = _rx(
    r"tri ?compartment", r"all three compartment", r"global(ised)? (oa|osteoarthrit)",
    r"\bgonarthros", r"\bgonartros", r"\bgonarthrose", r"\bgonartrose",
    r"osteoarthritis of the knee", r"artrosis (de |)(la )?rodilla", r"knee osteoarthrit",
    r"\bdiz osteoartrit", r"\bgonartroz", r"artroza koljena",
    r"οστεοαρθριτιδα", r"αρθριτιδα του γονατος",
    r"артроза на колянната", r"гонартроз",
    r"degenerative joint disease", r"\bdjd\b",
)

# A bare "bone marrow oedema" is not a contusion when it sits under a cartilage
# defect: subchondral oedema beneath a worn compartment is reactive degenerative signal,
# and reading it as a bruise turns every osteoarthritic knee into a trauma case.
DEGENERATIVE_MARROW = _rx(
    r"subchondral", r"subcondral", r"subkondral", r"supkondraln", r"subchondraln",
    r"υποχονδρι", r"субхондрал", r"subchondrale?",
    r"\bcyst", r"\bquist", r"\bzyste\b", r"\bcistic", r"reactive", r"reactivo",
)

TRAUMA = _rx(
    r"\bbruise\b", r"\bcontusion", r"\bkontuz", r"\bcontusion osea\b",
    r"\btrauma", r"\bimpaction\b", r"\bpivot shift\b", r"\bkissing\b",
    r"\bacute\b", r"\bagudo\b", r"\bakut", r"\bpivot kaymasi\b",
    r"\bcontusion osseuse\b", r"\bbone bruise\b", r"\bbotcontusie\b",
    r"\bконтузион", r"\bμωλωπ", r"\bkontuzij",
)


In [6]:
def extract(report: str) -> dict:
    """Extract twelve (score, confidence) pairs from one report."""
    cls = clauses(report)
    out = {}
    path_paired = _rx(TEAR.pattern, DEGEN.pattern, INJURY.pattern)

    for tgt in TARGETS:
        if tgt in PAIRED:
            s, c, npos, nneg = _score_clauses(cls, ANAT_MATCH[tgt], path_paired)
        elif tgt in OA_TARGETS:
            s, c, npos, nneg = _score_clauses(cls, COMPARTMENT_MATCH[tgt], OA_EVIDENCE)
        elif tgt == "Contusion":
            # Reactive subchondral oedema under a cartilage defect is osteoarthritis,
            # not a bruise. Explicit trauma wording pushes the other way.
            s, c, npos, nneg = _score_clauses(cls, DIRECT_MATCH[tgt], None, DECOY.get(tgt),
                                              context_penalty=DEGENERATIVE_MARROW,
                                              context_bonus=TRAUMA)
        else:
            s, c, npos, nneg = _score_clauses(cls, DIRECT_MATCH[tgt], None, DECOY.get(tgt))
        out[tgt] = s
        out[tgt + "__conf"] = c
        out[tgt + "__npos"] = npos
        out[tgt + "__nneg"] = nneg

    # --- cross-target corrections ------------------------------------------ #
    # A whole-joint osteoarthritis statement is evidence for every compartment that was
    # not separately assessed. Without this, "incipient OA of all three compartments"
    # scores zero on all three OA targets.
    g_hits = [c for c in cls if GLOBAL_OA.search(c) and _polarity(c, 0) == "positive"]
    if g_hits:
        gscore = 0.50 + 0.42 * max(_severity(c) for c in g_hits)
        for tgt in OA_TARGETS:
            if out[tgt + "__npos"] == 0 and out[tgt + "__nneg"] == 0:
                out[tgt] = max(out[tgt], gscore * 0.92)
                out[tgt + "__conf"] = max(out[tgt + "__conf"], 0.4)

    # Synovitis is frequently visible on the images and absent from the text, so silence
    # is weak evidence of absence here in a way it is not for other findings. Effusion is
    # its most reliable textual proxy - the two share a mechanism - so a silent synovitis
    # inherits a fraction of the effusion evidence instead of falling to the floor.
    if out["Synovitis__npos"] == 0 and out["Synovitis__nneg"] == 0:
        out["Synovitis"] = max(out["Synovitis"], 0.28 + 0.45 * (out["Effusion"] - 0.28))

    return out


def _severity(clause: str) -> float:
    """Weight one positive mention by how emphatic the sentence is.

    Ordered, not calibrated. A "moderate effusion" must outrank a "trace effusion" and
    both must outrank silence; the absolute numbers do not matter to AUC.
    """
    high = SEV_HIGH.search(clause) is not None
    low = SEV_LOW.search(clause) is not None
    if high and not low:
        return 1.0
    if low and not high:
        return 0.45
    return 0.75                       # unqualified mention


def _is_structure_heading(clause: str) -> bool:
    """A short line ending in a colon names a section; it asserts nothing by itself.

    clauses() emits both the heading joined to the value beneath it and the heading on its
    own. For the five targets whose anatomy word *is* the finding - effusion, synovitis,
    Baker's, contusion, fracture - the bare heading matches the term with no negation in
    scope, so a structured report reading `Fractures :` / `Aucune.` scored a confident
    positive off the heading while the joined clause correctly read the negation. Short
    headings are therefore skipped as assertions; the joined clause carries the meaning.
    """
    return clause.endswith(":") and len(clause.split()) <= 5


def _score_clauses(cls, anat_rx, path_rx=None, decoy_rx=None, context_penalty=None,
                   context_bonus=None):
    """Accumulate graded evidence over clauses for one target.

    Returns (score, confidence, n_pos, n_neg). Positives are graded by severity and by
    optional context regexes; negatives only matter when nothing positive was found,
    because reports assert normality for every structure they check.
    """
    n_pos = n_neg = n_unc = 0
    best = 0.0
    for c in cls:
        m = anat_rx.search(c)
        if not m:
            continue
        if decoy_rx is not None and decoy_rx.search(c):
            continue
        if path_rx is not None and not path_rx.search(c):
            if NORMALITY.search(c) and not NEGATION.search(c):
                n_neg += 1
            continue
        pol = _polarity(c, m.end())
        if pol == "positive" and _is_structure_heading(c):
            continue
        if pol == "positive":
            n_pos += 1
            w = _severity(c)
            if context_penalty is not None and context_penalty.search(c):
                w *= 0.45
            if context_bonus is not None and context_bonus.search(c):
                w = min(1.0, w * 1.35)
            best = max(best, w)
        elif pol == "negative":
            n_neg += 1
        else:
            n_unc += 1
            best = max(best, 0.30)

    if n_pos or n_unc:
        # 0.52 .. 0.95, ordered by the strongest single mention, nudged by repetition.
        score = min(0.95, 0.50 + 0.42 * best + 0.03 * min(n_pos, 3))
        conf = min(1.0, 0.55 + 0.15 * n_pos)
    elif n_neg:
        score = max(0.04, 0.20 - 0.04 * n_neg)
        conf = min(0.9, 0.45 + 0.12 * n_neg)
    else:
        score, conf = 0.28, 0.05          # silence sits above asserted-negative
    return score, conf, n_pos, n_neg

In [7]:
# https://www.kaggle.com/code/romanrozen/rsna-knee-data-structure-eda-baseline
def _polarity(clause: str, anchor_end: int) -> str:
    """Classify one clause as positive, negative or uncertain for a matched term.

    Scope is the whole clause. Clause segmentation already keeps statements short, and
    a window in characters mis-scopes badly across languages with different word orders -
    Turkish puts its negator at the end of the sentence, English at the front.
    """
    if UNCERTAIN.search(clause):
        return "uncertain"
    if NEGATION.search(clause):
        return "negative"
    if NORMALITY.search(clause):
        # "meniscus normal" negates; "normal ... but tear" does not.
        if TEAR.search(clause) or re.search(r"\bgrade [34]\b", clause):
            return "positive"
        return "negative"
    return "positive"


class _Matcher:
    """Phrase lexicon first, stem+side proximity as the fallback.

    Exposes `.search` so it drops into the same slot as a compiled pattern.
    """

    def __init__(self, phrase_rx, stem=None, side=None, window=55):
        self.phrase_rx = phrase_rx
        self.stem = stem
        self.side = side
        self.window = window

    def search(self, clause):
        m = self.phrase_rx.search(clause)
        if m is not None:
            return m
        if self.stem is not None and _near(clause, self.stem, self.side, self.window):
            return self.stem.search(clause)
        return None


ANAT_MATCH = {
    tgt: _Matcher(ANAT[tgt], *STEM_RULES[tgt]) for tgt in PAIRED
}
COMPARTMENT_MATCH = {
    "Medial OA": _Matcher(COMPARTMENT["Medial OA"], *STEM_RULES["Medial OA"]),
    "Lateral OA": _Matcher(COMPARTMENT["Lateral OA"], *STEM_RULES["Lateral OA"]),
    "PF OA": _Matcher(COMPARTMENT["PF OA"]),
}
DIRECT_MATCH = {
    tgt: _Matcher(_rx(rx.pattern, STEM_FRACTURE.pattern) if tgt == "Fracture" else rx)
    for tgt, rx in DIRECT.items()
}



In [8]:
# Additions are unioned onto the compiled patterns rather than edited into them, so the
# diff against the original lexicon stays readable and every entry can be reverted alone.

NEGATION = _rx(
    NEGATION.pattern,
    r"\bnone\b", r"\bnil\b",                       # "Effusion: none."
    r"not (identified|seen|visuali[sz]ed|demonstrated|detected|present|appreciated)",
    r"\bnegative\b",
    r"\bningun[ao]?\b", r"no se (identifica|aprecia|visualiza|reconoce)",
    r"\bnenhum\w*",
    r"niet zichtbaar", r"\bafwezigheid\b",
    r"izlenmemistir", r"gorulmemistir",
    # Rejected: r"\bnon\b". It negates "fracture non deplacee", which is a
    # non-displaced fracture - a positive finding with a qualifier, not an absence.
)

NORMALITY = _rx(
    NORMALITY.pattern,
    r"\bwnl\b", r"sans particularite", r"sin particularidades",
)

# A decoy skips the clause; a negation scores it. "no fracture" belongs to the second.
DECOY["Fracture"] = _rx(r"microfractur", r"\bfracture (risk|prophyla)")

_checks = {
    ("Fracture", "Fractures :\nAucune."): "lt",
    ("Fracture", "No fracture is identified."): "lt",
    ("Fracture", "Fractures :\nFracture non deplacee du plateau tibial."): "gt",
    ("Fracture", "Microfracture of the trochlea was performed."): "silent",
    ("Effusion", "Effusion:\nNone."): "lt",
    ("Effusion", "Joint effusion: nil."): "lt",
    ("Effusion", "Moderate joint effusion."): "gt",
    ("Fracture", "Fracturas :\nNinguna."): "lt",
    ("Baker's", "Quiste de Baker :\nPresente."): "gt",
}
for (tgt, text), want in _checks.items():
    v = extract(text)[tgt]
    ok = (v < 0.3) if want == "lt" else (v > 0.5) if want == "gt" else (0.25 < v < 0.32)
    assert ok, f"{want}: {tgt} scored {v:.2f} on {text!r}"
print("lexicon repairs verified on 9 regression cases")

lexicon repairs verified on 9 regression cases


In [9]:
lab = pd.DataFrame([extract(r) for r in train["Report"].fillna("")])
lab.index = train["StudyInstanceUID"].values
Y_soft = lab[TARGETS].values.astype(np.float32)
log(f"derived targets {Y_soft.shape}, positive rate {(Y_soft>0.5).mean():.3f}")

[  24.0s] derived targets (4407, 12), positive rate 0.252


In [10]:
print(extract("No fracture is identified.")["Fracture"])          # ~0.16 (low)
print(extract("Moderate joint effusion.")["Effusion"])            # ~0.95 (high)
print(extract("Ρηξη του εσω μηνισκου.")["Medial Meniscus"])       # >0.5

0.16
0.95
0.845


In [11]:
def composition_features(sdf):
    sdf = sdf.copy()
    for c in ("Fluid_Sensitive", "Fat_Suppression"):
        sdf[c] = pd.to_numeric(sdf[c], errors="coerce").fillna(0).astype(int)
    sdf["combo"] = (sdf["Anatomical_Plane"].astype(str)
                    + "_F" + sdf["Fluid_Sensitive"].astype(str)
                    + "_S" + sdf["Fat_Suppression"].astype(str))
    piv = pd.crosstab(sdf["StudyInstanceUID"], sdf["combo"])
    piv.columns = [f"n_{c}" for c in piv.columns]
    piv["n_series"] = piv.sum(axis=1)
    sig = sdf.groupby("StudyInstanceUID")["combo"].apply(
        lambda s: "|".join(sorted(set(s))))
    piv["protocol_sig"] = sig.reindex(piv.index)   # align on index, not position
    return piv

comp = composition_features(train_series).reindex(train["StudyInstanceUID"].values)
log(f"tier-0 features {comp.shape}")
print(f"distinct protocol signatures: {comp['protocol_sig'].nunique()}")
print(comp["protocol_sig"].value_counts().head(15))

[  24.5s] tier-0 features (4407, 8)
distinct protocol signatures: 12
protocol_sig
Axial_F1_S1|Coronal_F0_S0|Coronal_F1_S1|Sagittal_F0_S0|Sagittal_F1_S1                2516
Axial_F1_S1|Coronal_F1_S1|Sagittal_F0_S0|Sagittal_F1_S1                               745
Axial_F0_S0|Axial_F1_S1|Coronal_F0_S0|Coronal_F1_S1|Sagittal_F0_S0|Sagittal_F1_S1     566
Axial_F0_S0|Axial_F1_S1|Coronal_F1_S1|Sagittal_F0_S0                                  229
Axial_F1_S1|Coronal_F0_S0|Sagittal_F0_S0|Sagittal_F1_S1                               157
Axial_F1_S1|Coronal_F0_S0|Coronal_F1_S1|Sagittal_F1_S1                                120
Axial_F0_S0|Axial_F1_S1|Coronal_F1_S1|Sagittal_F0_S0|Sagittal_F1_S1                    23
Axial_F0_S0|Axial_F1_S1|Coronal_F0_S0|Coronal_F1_S1|Sagittal_F1_S1                     21
Axial_F0_S0|Axial_F1_S1|Coronal_F0_S0|Coronal_F1_S1|Sagittal_F0_S0                     16
Axial_F1_S1|Coronal_F0_S0|Coronal_F1_S1|Sagittal_F0_S0                                  8
Axial_F1_S1|Corona

In [12]:
def prep(df):
    """Object columns -> integer codes; returns matrix and categorical mask."""
    X = df.copy()
    cat = []
    for c in X.columns:
        if X[c].dtype == object:
            X[c] = pd.factorize(X[c])[0]
            cat.append(True)
        else:
            X[c] = pd.to_numeric(X[c], errors="coerce")
            cat.append(False)
    return X.reset_index(drop=True), np.array(cat)

def run_probe(df, Y_soft, groups=None, n_splits=5, seed=0, label=""):
    X, cat = prep(df)
    y = (Y_soft > 0.5).astype(int)
    oof = np.full(y.shape, np.nan)

    for j in range(y.shape[1]):
        if len(np.unique(y[:, j])) < 2:
            continue
        if groups is None:
            folds = StratifiedKFold(n_splits, shuffle=True,
                                    random_state=seed).split(X, y[:, j])
        else:
            folds = GroupKFold(n_splits=n_splits).split(X, y[:, j], groups)
        for tr, va in folds:
            if len(np.unique(y[tr, j])) < 2:
                continue
            m = HistGradientBoostingClassifier(
                max_iter=250, learning_rate=0.06, max_depth=6,
                categorical_features=cat, random_state=seed)
            m.fit(X.iloc[tr], y[tr, j])
            oof[va, j] = m.predict_proba(X.iloc[va])[:, 1]

    aucs = {}
    for j, t in enumerate(TARGETS):
        ok = ~np.isnan(oof[:, j])
        aucs[t] = (roc_auc_score(y[ok, j], oof[ok, j])
                   if ok.sum() and len(np.unique(y[ok, j])) > 1 else np.nan)
    s = pd.Series(aucs)
    print(f"\n=== {label} ===")
    print(s.round(3).to_string())
    print(f"MACRO AUC: {np.nanmean(s):.4f}")
    return s, oof

In [13]:
s0, oof0 = run_probe(comp, Y_soft, label="Tier-0: series composition only")


=== Tier-0: series composition only ===
ACL                 0.663
MCL                 0.616
Medial Meniscus     0.554
Lateral Meniscus    0.582
Medial OA           0.542
Lateral OA          0.539
PF OA               0.580
Effusion            0.612
Synovitis           0.623
Baker's             0.669
Contusion           0.606
Fracture            0.558
MACRO AUC: 0.5954


In [14]:
HDR = [
    # scanner identity — the fingerprint
    "Manufacturer","ManufacturerModelName","SoftwareVersions",
    "MagneticFieldStrength","ImagingFrequency",
    "ReceiveCoilName","TransmitCoilName",
    # protocol free text
    "SeriesDescription","SequenceName","ScanOptions","ScanningSequence",
    "SequenceVariant","MRAcquisitionType","ImageType","BodyPartExamined",
    # acquisition parameters
    "RepetitionTime","EchoTime","InversionTime","FlipAngle","EchoTrainLength",
    "PixelBandwidth","NumberOfAverages","NumberOfPhaseEncodingSteps",
    "PercentSampling","PercentPhaseFieldOfView","AcquisitionMatrix",
    "SliceThickness","SpacingBetweenSlices","PixelSpacing","Rows","Columns",
    "VariableFlipAngleFlag","ContrastBolusAgent",
    # display + patient
    "WindowCenter","WindowWidth","PatientSex","PatientID",
    "Laterality","PatientPosition","SeriesNumber",
]

def probe_series(item):
    study, series, path = item
    row = {"StudyInstanceUID": study, "SeriesInstanceUID": series}
    try:
        files = [e.name for e in os.scandir(path) if e.name.endswith(".dcm")]
        row["n_slices"] = len(files)
        if not files: return row
        ds = pydicom.dcmread(os.path.join(path, sorted(files)[len(files)//2]),
                             stop_before_pixels=True, force=True)
        for t in HDR:
            v = getattr(ds, t, None)
            if v is None: row[t] = None
            elif isinstance(v, (list, tuple)) or type(v).__name__ == "MultiValue":
                row[t] = "|".join(str(x) for x in v)
            else: row[t] = str(v)
    except Exception as e:
        row["err"] = str(e)[:80]
    return row

items = [(st.name, se.name, se.path)
         for st in os.scandir(ROOT/"train_series") if st.is_dir()
         for se in os.scandir(st.path) if se.is_dir()]
log(f"reading {len(items)} series headers")
with ThreadPoolExecutor(max_workers=16) as pool:
    H = pd.DataFrame(list(pool.map(probe_series, items)))
log(f"done: {H.shape}")
H.to_parquet("/kaggle/working/headers.parquet")   # cache so you never redo this

[  71.5s] reading 24371 series headers
[ 195.7s] done: (24371, 43)


In [15]:
NUM = ["RepetitionTime","EchoTime","InversionTime","FlipAngle","EchoTrainLength",
       "PixelBandwidth","NumberOfAverages","NumberOfPhaseEncodingSteps",
       "PercentSampling","PercentPhaseFieldOfView","SliceThickness",
       "SpacingBetweenSlices","Rows","Columns","MagneticFieldStrength",
       "ImagingFrequency","n_slices"]
CATS = ["Manufacturer","ManufacturerModelName","SoftwareVersions",
        "ReceiveCoilName","TransmitCoilName","PatientSex",
        "BodyPartExamined","MRAcquisitionType","PatientPosition"]

for c in NUM:
    H[c] = pd.to_numeric(H.get(c), errors="coerce")
H["px"] = pd.to_numeric(H["PixelSpacing"].fillna("").str.split("|").str[0]
                        .replace("", np.nan), errors="coerce")

g = H.groupby("StudyInstanceUID")
agg = g[NUM + ["px"]].agg(["median","min","max"])
agg.columns = [f"{a}_{b}" for a, b in agg.columns]
agg["total_slices"] = g["n_slices"].sum()

for c in CATS:
    if c in H.columns:
        agg[c] = g[c].agg(lambda s: s.dropna().mode().iloc[0] if s.dropna().size else None)

feat = agg.join(comp, how="right").reindex(train["StudyInstanceUID"].values)
log(f"full feature table {feat.shape}")
print("PatientSex coverage:", feat["PatientSex"].notna().mean().round(3))

[ 210.0s] full feature table (4407, 72)
PatientSex coverage: 0.946


### Probe A

In [16]:
finger = (feat["Manufacturer"].astype(str) + "|"
          + feat["ManufacturerModelName"].astype(str) + "|"
          + feat["SoftwareVersions"].astype(str) + "|"
          + feat["ImagingFrequency_median"].round(3).astype(str) + "|"
          + feat["ReceiveCoilName"].astype(str))
site = pd.factorize(finger)[0]

vc = pd.Series(site).value_counts()
print(f"distinct scanner fingerprints: {len(vc)}")
print(f"top 20 cover {vc.head(20).sum()/len(site):.1%} of studies")
print(vc.head(20))

distinct scanner fingerprints: 265
top 20 cover 45.5% of studies
53    245
26    204
6     129
65    120
18    113
20    110
13    108
27    107
16    103
14     96
3      92
70     76
1      72
23     71
92     69
47     66
21     59
57     57
4      55
39     53
Name: count, dtype: int64


### Probe B

In [17]:
s_rand, oof_r = run_probe(feat, Y_soft, label="Full metadata — RANDOM folds")
s_site, oof_s = run_probe(feat, Y_soft, groups=site, label="Full metadata — SITE-GROUPED folds")

cmp = pd.DataFrame({"random": s_rand, "site_grouped": s_site})
cmp["drop"] = cmp["random"] - cmp["site_grouped"]
print(cmp.round(3).to_string())
print(f"\nmacro random {np.nanmean(s_rand):.4f} | "
      f"site-grouped {np.nanmean(s_site):.4f} | "
      f"gap {np.nanmean(s_rand)-np.nanmean(s_site):.4f}")


=== Full metadata — RANDOM folds ===
ACL                 0.705
MCL                 0.683
Medial Meniscus     0.590
Lateral Meniscus    0.595
Medial OA           0.652
Lateral OA          0.637
PF OA               0.680
Effusion            0.628
Synovitis           0.644
Baker's             0.765
Contusion           0.633
Fracture            0.605
MACRO AUC: 0.6515

=== Full metadata — SITE-GROUPED folds ===
ACL                 0.670
MCL                 0.648
Medial Meniscus     0.548
Lateral Meniscus    0.565
Medial OA           0.578
Lateral OA          0.563
PF OA               0.599
Effusion            0.582
Synovitis           0.602
Baker's             0.717
Contusion           0.587
Fracture            0.519
MACRO AUC: 0.5981
                  random  site_grouped   drop
ACL                0.705         0.670  0.035
MCL                0.683         0.648  0.035
Medial Meniscus    0.590         0.548  0.042
Lateral Meniscus   0.595         0.565  0.030
Medial OA          0.652    

### Footnote

In [18]:
gi = train.index[train[TARGETS].notna().all(axis=1)].values
yg = train.loc[gi, TARGETS].values.astype(int)
for nm, oof in (("random", oof_r), ("site-grouped", oof_s)):
    a = [roc_auc_score(yg[:, j], oof[gi, j])
         for j in range(12) if len(np.unique(yg[:, j])) > 1]
    print(f"{nm}: macro AUC vs 58 gold = {np.mean(a):.3f}")

random: macro AUC vs 58 gold = 0.560
site-grouped: macro AUC vs 58 gold = 0.589


In [19]:
# ---- test side: same header pass, same feature construction -------------------
te_items = [(st.name, se.name, se.path)
            for st in os.scandir(ROOT/"test_series") if st.is_dir()
            for se in os.scandir(st.path) if se.is_dir()]
log(f"reading {len(te_items)} test series headers")
with ThreadPoolExecutor(max_workers=16) as pool:
    Hte = pd.DataFrame(list(pool.map(probe_series, te_items)))
log(f"test headers {Hte.shape}")

def build_features(H_, series_df, index_uids):
    H_ = H_.copy()
    for c in NUM:
        H_[c] = pd.to_numeric(H_.get(c), errors="coerce")
    H_["px"] = pd.to_numeric(H_["PixelSpacing"].fillna("").str.split("|").str[0]
                             .replace("", np.nan), errors="coerce")
    g_ = H_.groupby("StudyInstanceUID")
    a = g_[NUM + ["px"]].agg(["median","min","max"])
    a.columns = [f"{x}_{y}" for x, y in a.columns]
    a["total_slices"] = g_["n_slices"].sum()
    for c in CATS:
        if c in H_.columns:
            a[c] = g_[c].agg(lambda s: s.dropna().mode().iloc[0]
                             if s.dropna().size else None)
    return a.join(composition_features(series_df), how="right").reindex(index_uids)

feat_te = build_features(Hte, test_series, test["StudyInstanceUID"].values)
log(f"test features {feat_te.shape}")

[ 365.4s] reading 15 test series headers
[ 365.5s] test headers (15, 43)
[ 365.6s] test features (3, 72)


In [20]:
# ---- fit on all of train, predict test ---------------------------------------
both = pd.concat([feat, feat_te], axis=0, ignore_index=True)
Xb, cat = prep(both)                       # one shared factorization
Xtr, Xte = Xb.iloc[:len(feat)], Xb.iloc[len(feat):]
y = (Y_soft > 0.5).astype(int)

pred = np.full((len(feat_te), len(TARGETS)), 0.5, dtype=np.float32)
for j, t in enumerate(TARGETS):
    if len(np.unique(y[:, j])) < 2:
        continue
    m = HistGradientBoostingClassifier(max_iter=250, learning_rate=0.06, max_depth=6,
                                       categorical_features=cat, random_state=0)
    m.fit(Xtr, y[:, j])
    pred[:, j] = m.predict_proba(Xte)[:, 1]
    log(f"{t}: mean pred {pred[:, j].mean():.3f}")

sub = pd.DataFrame(pred, columns=TARGETS)
sub.insert(0, "StudyInstanceUID", test["StudyInstanceUID"].values)
sub.to_csv("submission.csv", index=False)
print(sub.shape); print(sub.head())

[ 367.1s] ACL: mean pred 0.295
[ 368.5s] MCL: mean pred 0.128
[ 370.0s] Medial Meniscus: mean pred 0.297
[ 371.2s] Lateral Meniscus: mean pred 0.158
[ 372.5s] Medial OA: mean pred 0.178
[ 373.7s] Lateral OA: mean pred 0.185
[ 375.0s] PF OA: mean pred 0.435
[ 376.3s] Effusion: mean pred 0.485
[ 377.5s] Synovitis: mean pred 0.422
[ 379.2s] Baker's: mean pred 0.212
[ 380.6s] Contusion: mean pred 0.383
[ 382.1s] Fracture: mean pred 0.057
(3, 13)
                                    StudyInstanceUID       ACL       MCL  \
0  1.2.826.0.1.3680043.8.498.10047035057544427318...  0.114748  0.035659   
1  1.2.826.0.1.3680043.8.498.10062861783145312629...  0.727546  0.268670   
2  1.2.826.0.1.3680043.8.498.10067514707072572280...  0.042090  0.078900   

   Medial Meniscus  Lateral Meniscus  Medial OA  Lateral OA     PF OA  \
0         0.126764          0.188929   0.125604    0.079236  0.336734   
1         0.542945          0.267925   0.227928    0.142897  0.432993   
2         0.222763          0.